In [96]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_bounds
import numpy as np
import os
import pandas as pd
import re

In [98]:
folder = '/home/jan/Documents/EPFL/project_2/data/input_datasets/manual_annotation'
gdf_coco_colombia = gpd.read_file(os.path.join(folder, 'cocoa_colombia_manual.gpkg'))
gdf_cocoa_other = gpd.read_file(os.path.join(folder, 'manual_cocoa.gpkg'))

gdf_coffee_colombia = gpd.read_file(os.path.join(folder, 'coffee_colombia_manual.gpkg'))
gdf_coffee_eth = gpd.read_file(os.path.join(folder, 'coffee_ethiopia_manual.gpkg'))
gdf_coffee_main = gpd.read_file(os.path.join(folder, 'coffee_manual_CROPGRIDS.gpkg'))

gdf_rubber = gpd.read_file(os.path.join(folder, 'rubber_manual.gpkg'))



In [99]:
gdf_cocoa_other

,geometry
0,"POLYGON ((152.18384 -4.42242, 152.18361 -4.423..."
1,"POLYGON ((152.18463 -4.43183, 152.18665 -4.431..."
2,"POLYGON ((152.18476 -4.43647, 152.18302 -4.434..."
3,"POLYGON ((152.18229 -4.44031, 152.18365 -4.441..."
4,"POLYGON ((152.18277 -4.45865, 152.18303 -4.458..."
5,"POLYGON ((152.18656 -4.46191, 152.18714 -4.461..."
6,"POLYGON ((152.17778 -4.48500, 152.17889 -4.482..."
7,"POLYGON ((152.16134 -4.49418, 152.16332 -4.495..."
8,"POLYGON ((152.17306 -4.47182, 152.17268 -4.472..."
9,"POLYGON ((152.26374 -4.45211, 152.26409 -4.450..."


In [100]:
gdf_coffee_colombia['driver'] = 1
gdf_coffee_eth['driver'] = 1
gdf_coffee_main['driver'] = 1

gdf_cocoa_other['driver'] = 2
gdf_coco_colombia['driver'] = 2

gdf_rubber['driver'] = 3

In [101]:
gdf = gpd.GeoDataFrame(pd.concat([gdf_coffee_colombia, gdf_coffee_eth, gdf_coffee_main, gdf_coco_colombia, gdf_rubber, gdf_cocoa_other])).reset_index(drop=True)

In [102]:
gdf

,geometry,driver
0,"POLYGON ((-76.17559 1.76667, -76.17468 1.76724...",1
1,"POLYGON ((-76.17395 1.76911, -76.17372 1.76932...",1
2,"POLYGON ((-76.17856 1.77174, -76.17791 1.77155...",1
3,"POLYGON ((-76.18027 1.76984, -76.17950 1.77007...",1
4,"POLYGON ((-76.18236 1.76908, -76.18216 1.76926...",1
...,...,...
487,"POLYGON ((152.27414 -4.47314, 152.27699 -4.472...",2
488,"POLYGON ((152.27238 -4.47341, 152.27375 -4.473...",2
489,"POLYGON ((152.19405 -4.48257, 152.19489 -4.479...",2
490,"POLYGON ((152.17111 -4.49361, 152.19244 -4.512...",2


In [103]:

def split_polygons_by_country(gdf, countries_gdf):
    # Ensure CRS match
    gdf = gdf.to_crs(countries_gdf.crs)
    
    # Create a dictionary to hold subsets
    country_polygons = {country: [] for country in countries_gdf['name']}
    
    for _, polygon in gdf.iterrows():
        for _, country in countries_gdf.iterrows():
            if polygon.geometry.intersects(country.geometry):
                country_polygons[country['name']].append(polygon)
                break

    # Convert lists to GeoDataFrames
    for country in country_polygons:
        if country_polygons[country]:
            country_polygons[country] = gpd.GeoDataFrame(country_polygons[country], columns=gdf.columns, crs=gdf.crs)
        else:
            country_polygons[country] = gpd.GeoDataFrame(columns=gdf.columns, crs=gdf.crs)

    return country_polygons



def rasterize_polygons(gdf, out_path, resolution=10, max_size=10000):
    
    gdf = gdf.to_crs(gdf.estimate_utm_crs())

    
    # Check if there are any geometries to rasterize
    if gdf.empty or gdf.geometry.is_empty.all():
        print(f"Skipping {out_path} as it is empty.")
        return
    
    # Get the bounds of the GeoDataFrame
    minx, miny, maxx, maxy = gdf.total_bounds

    # Calculate the dimensions of the raster based on the resolution
    width = int((maxx - minx) / resolution)
    height = int((maxy - miny) / resolution)

    # Skip rasterization if width or height is zero
    if width <= 0 or height <= 0:
        print(f"Skipping {out_path} due to invalid dimensions (width: {width}, height: {height}).")
        return

    # Create a transform for the output raster
    transform = from_bounds(minx, miny, maxx, maxy, width, height)

    # Rasterize the GeoDataFrame
    raster = rasterize(
        [(geom, value) for geom, value in zip(gdf.geometry, gdf['driver'])],
        out_shape=(height, width),
        transform=transform,
        fill=0,
        dtype='uint8'
    )

    # Split large rasters if necessary
    if width > max_size or height > max_size:
        x_splits = max(1, width // max_size + (1 if width % max_size != 0 else 0))
        y_splits = max(1, height // max_size + (1 if height % max_size != 0 else 0))
        for i in range(x_splits):
            for j in range(y_splits):
                x_min = i * max_size
                x_max = min((i + 1) * max_size, width)
                y_min = j * max_size
                y_max = min((j + 1) * max_size, height)
                
                sub_raster = raster[y_min:y_max, x_min:x_max]
                sub_transform = transform * transform.translation(x_min, y_min)
                
                sub_out_path = f"{os.path.splitext(out_path)[0]}_{i}_{j}.tif"
                if np.any(sub_raster):
                    with rasterio.open(
                        sub_out_path,
                        'w',
                        driver='GTiff',
                        height=sub_raster.shape[0],
                        width=sub_raster.shape[1],
                        count=1,
                        dtype=sub_raster.dtype,
                        crs=gdf.crs,
                        transform=sub_transform,
                    ) as dst:
                        dst.write(sub_raster, 1)
                    print(f"Written {sub_out_path}")
    else:
        if np.any(raster):
            with rasterio.open(
                out_path,
                'w',
                driver='GTiff',
                height=raster.shape[0],
                width=raster.shape[1],
                count=1,
                dtype=raster.dtype,
                crs=gdf.crs,
                transform=transform,
            ) as dst:
                dst.write(raster, 1)
            print(f"Written {out_path}")


        


def sanitize_filename(name):
    # Replace or remove any problematic characters for filenames
    name = re.sub(r'[^\w\s-]', '', name)
    name = re.sub(r'[-\s]+', '_', name)
    return name

In [104]:

world_filepath = gpd.datasets.get_path('naturalearth_lowres')
world = gpd.read_file(world_filepath)

/tmp/ipykernel_2790741/947498767.py:1: FutureWarning: The geopandas.dataset module is deprecated and will be removed in GeoPandas 1.0. You can get the original 'naturalearth_lowres' data from https://www.naturalearthdata.com/downloads/110m-cultural-vectors/.
  world_filepath = gpd.datasets.get_path('naturalearth_lowres')


In [105]:

# Example usage
# polygons_gdf: GeoDataFrame of polygons to split
# countries_gdf: GeoDataFrame of countries with a 'name' column
country_subsets = split_polygons_by_country(gdf, world)



In [106]:
country_subsets_f = {k:v for k,v in country_subsets.items() if not v.empty}

In [107]:
# Rasterize each subset
for country, subset_gdf in country_subsets_f.items():
    print(country)
    sanitized_country_name = sanitize_filename(country)
    rasterize_polygons(subset_gdf, f'../data/input_datasets/manual_annotation/rasters/{sanitized_country_name}.tif')

Papua New Guinea
Written ../data/input_datasets/manual_annotation/rasters/Papua_New_Guinea.tif
Colombia
Written ../data/input_datasets/manual_annotation/rasters/Colombia_0_6.tif
Written ../data/input_datasets/manual_annotation/rasters/Colombia_1_5.tif
Written ../data/input_datasets/manual_annotation/rasters/Colombia_4_0.tif
Written ../data/input_datasets/manual_annotation/rasters/Colombia_6_0.tif
Written ../data/input_datasets/manual_annotation/rasters/Colombia_7_0.tif
Costa Rica
Written ../data/input_datasets/manual_annotation/rasters/Costa_Rica.tif
Honduras
Written ../data/input_datasets/manual_annotation/rasters/Honduras.tif
Venezuela
Written ../data/input_datasets/manual_annotation/rasters/Venezuela.tif
Puerto Rico
Written ../data/input_datasets/manual_annotation/rasters/Puerto_Rico.tif
Cameroon
Written ../data/input_datasets/manual_annotation/rasters/Cameroon.tif
Côte d'Ivoire
Written ../data/input_datasets/manual_annotation/rasters/Côte_dIvoire_0_0.tif
Written ../data/input_datas

In [108]:
import sys
sys.path.append('../')

from utils import reproject_raster

import glob

for raster in glob.glob('../data/input_datasets/manual_annotation/rasters/*.tif'):
    out_raster = raster.replace('rasters', 'rasters_fixed')
    reproject_raster(raster, out_raster, dst_crs='EPSG:4326')